# HackerNews Analytics Pro v6.0 — Story Intelligence

This notebook analyses repeated SQLite snapshots: score velocity, comment velocity, rank movement and rising/cooling story status. It does not infer causality or forecast future popularity.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT).exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from hn_analytics import story_momentum, export_momentum
DB=ROOT/'hn_monitor.sqlite'


## 1. Load the monitoring history
A database is created by the v5/v6 monitor command. Rank tracking is available for snapshots recorded by v6.


In [ ]:
momentum=story_momentum(DB, lookback_hours=24)
momentum.head(20)


## 2. Compare score and comment velocity
Velocity is the observed change divided by elapsed hours.


## 4. Three-dimensional momentum comparison
Axes show score velocity, comment velocity and rank improvement. Rank values may be missing for legacy v5 snapshots.


In [ ]:
if not momentum.empty:
    plot_data=momentum.dropna(subset=['score_per_hour','comments_per_hour']).copy()
    if not plot_data.empty:
        fig3d=px.scatter_3d(plot_data, x='score_per_hour', y='comments_per_hour', z='rank_improvement',
                            color='momentum_status', hover_name='title',
                            title='Story momentum: score velocity × comment velocity × rank movement')
        fig3d.show()


In [ ]:
if not momentum.empty:
    display(momentum[['title','score_gain','score_per_hour','comment_gain','comments_per_hour','rank_improvement','momentum_status']].head(20))


## 3. Visualize top observed velocity


In [ ]:
import plotly.express as px
if not momentum.empty:
    fig=px.bar(momentum.head(15), x='score_per_hour', y='title', orientation='h', title='Top observed score velocity (points/hour)')
    fig.update_layout(yaxis={'categoryorder':'total ascending'})
    fig.show()


## 5. Export reusable results


In [ ]:
if DB.exists():
    export_momentum(DB, ROOT/'reports', lookback_hours=24)


## Interpretation notes
- Rising/cooling labels describe only the observed interval.
- A positive rank improvement means movement closer to rank 1.
- Zero-duration comparisons have undefined velocity.
- Missing legacy rank values are expected for snapshots collected before v6.
- Feed membership is bounded and changes over time; this is not a complete HN archive.
